In [1]:
import sys
sys.path.insert(0, "../src")

import json
import time
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score

from finnews_sentiment.models.train_model import build_baseline_pipeline

In [2]:
train_clean = pd.read_csv("data/train_clean.csv").dropna(subset=["text_clean"])
test_clean = pd.read_csv("data/test_clean.csv").dropna(subset=["text_clean"])

X_train, X_valid, y_train, y_valid = train_test_split(
    train_clean["text_clean"], train_clean["label"],
    test_size=0.2, random_state=42, stratify=train_clean["label"]
)
print(len(X_train), len(X_valid))

95240 23810


In [3]:
results = []

def run(name, pipe, X_tr=X_train, X_va=X_valid):
    start = time.time()
    pipe.fit(X_tr, y_train)
    pred = pipe.predict(X_va)
    results.append({
        "experiment": name,
        "accuracy": round(accuracy_score(y_valid, pred), 4),
        "macro_f1": round(f1_score(y_valid, pred, average="macro"), 4),
        "time_sec": round(time.time() - start, 1),
    })
    print(results[-1])
    return pipe

In [4]:
run("0. baseline: tfidf 5000, 1-2gram + logreg", build_baseline_pipeline());

/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


{'experiment': '0. baseline: tfidf 5000, 1-2gram + logreg', 'accuracy': 0.9059, 'macro_f1': 0.9057, 'time_sec': 9.6}


In [5]:
run("1. tfidf 50000 + sublinear + logreg", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000)),
]));

{'experiment': '1. tfidf 50000 + sublinear + logreg', 'accuracy': 0.9173, 'macro_f1': 0.9171, 'time_sec': 11.2}


In [6]:
run("2a. tfidf 50000 + linear svm", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC()),
]));

run("2b. tfidf 50000 + complement nb", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", ComplementNB()),
]));

{'experiment': '2a. tfidf 50000 + linear svm', 'accuracy': 0.9197, 'macro_f1': 0.9196, 'time_sec': 12.2}
{'experiment': '2b. tfidf 50000 + complement nb', 'accuracy': 0.9087, 'macro_f1': 0.9082, 'time_sec': 8.0}


In [7]:
X_train_raw = train_clean.loc[X_train.index, "text"].str.lower()
X_valid_raw = train_clean.loc[X_valid.index, "text"].str.lower()

run("3. raw text + tfidf 50000 + linear svm", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC()),
]), X_tr=X_train_raw, X_va=X_valid_raw);

{'experiment': '3. raw text + tfidf 50000 + linear svm', 'accuracy': 0.9222, 'macro_f1': 0.922, 'time_sec': 15.9}


In [8]:
for c in [0.1, 0.3, 1, 3]:
    run(f"4. tfidf 50000 + linear svm, C={c}", Pipeline([
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", LinearSVC(C=c)),
    ]));

{'experiment': '4. tfidf 50000 + linear svm, C=0.1', 'accuracy': 0.9187, 'macro_f1': 0.9184, 'time_sec': 11.2}
{'experiment': '4. tfidf 50000 + linear svm, C=0.3', 'accuracy': 0.9227, 'macro_f1': 0.9225, 'time_sec': 8.7}
{'experiment': '4. tfidf 50000 + linear svm, C=1', 'accuracy': 0.9197, 'macro_f1': 0.9196, 'time_sec': 9.1}
{'experiment': '4. tfidf 50000 + linear svm, C=3', 'accuracy': 0.9113, 'macro_f1': 0.9111, 'time_sec': 9.7}


In [9]:
res = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
res["vs_baseline"] = (res["macro_f1"] - res.loc[res["experiment"].str.startswith("0."), "macro_f1"].values[0]).round(4)
res

,experiment,accuracy,macro_f1,time_sec,vs_baseline
6,"4. tfidf 50000 + linear svm, C=0.3",0.9227,0.9225,8.7,0.0168
4,3. raw text + tfidf 50000 + linear svm,0.9222,0.9220,15.9,0.0163
2,2a. tfidf 50000 + linear svm,0.9197,0.9196,12.2,0.0139
7,"4. tfidf 50000 + linear svm, C=1",0.9197,0.9196,9.1,0.0139
5,"4. tfidf 50000 + linear svm, C=0.1",0.9187,0.9184,11.2,0.0127
1,1. tfidf 50000 + sublinear + logreg,0.9173,0.9171,11.2,0.0114
8,"4. tfidf 50000 + linear svm, C=3",0.9113,0.9111,9.7,0.0054
3,2b. tfidf 50000 + complement nb,0.9087,0.9082,8.0,0.0025
0,"0. baseline: tfidf 5000, 1-2gram + logreg",0.9059,0.9057,9.6,0.0000


In [10]:
best = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(C=0.3)),
])
best.fit(train_clean["text_clean"], train_clean["label"])
test_pred = best.predict(test_clean["text_clean"])

with open("../results/baseline_metrics.json") as f:
    baseline = json.load(f)

best_results = {
    "model": "tfidf 50000 1-2gram sublinear + linear svm C=0.3",
    "test_accuracy": round(accuracy_score(test_clean["label"], test_pred), 4),
    "test_macro_f1": round(f1_score(test_clean["label"], test_pred, average="macro"), 4),
}
print("baseline:", baseline["test_macro_f1"], "-> best:", best_results["test_macro_f1"])

baseline: 0.9019 -> best: 0.9217


In [11]:
res.to_csv("../results/day04_experiments.csv", index=False)
with open("../results/best_metrics.json", "w") as f:
    json.dump(best_results, f, indent=2)
joblib.dump({"model": best, "meta": best_results}, "../models/best.joblib")

['../models/best.joblib']

## Выводы

- baseline (tfidf 5000 + logreg): valid macro F1 = 0.9057
- больше признаков (50000) + sublinear_tf + min_df=2: 0.9171, +1.1 п.п. - главный прирост, 5000 признаков было мало
- linear svm вместо logreg: 0.9196, ещё +0.25 п.п.
- complement nb: 0.9082 - слабее, но самый быстрый
- подбор C для svm: лучший C=0.3 (0.9225), при C=3 качество падает до 0.9111 - слабая регуляризация, модель переобучается
- сырой текст без лемматизации и стоп-слов: 0.9220 против 0.9196 на очищенном при тех же настройках - агрессивная очистка для tfidf не помогла
- лучший вариант: tfidf 50000 1-2gram sublinear + linear svm C=0.3, итог +1.7 п.п. к baseline на valid
- на test: macro F1 = ... против ... у baseline
- разница между лучшими вариантами 0.2-0.3 п.п., это примерно уровень шума на 24 тыс. объектов